In [1]:
# Import libraries
import pandas as pd 
import numpy as np 
import math 
import statistics 
import matplotlib.pyplot as plt 

#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)

# Need to reindex to trading days 
import pandas_market_calendars as mcal

In [3]:
# Import Q2_2026 data
raw_df = pd.read_csv(r".\Signal_data\Q2_2026-06-02.csv", engine="python") 

# Drop non-ES columns
raw_df_2 = raw_df[["utc_date", "ES_classification", "ES_refOpen"]].copy()

# Add Quarter column
raw_df_2["Quarter"] = "2020Q2"
raw_df_2 = raw_df_2[["utc_date", "Quarter", "ES_classification", "ES_refOpen"]]

# Rename for convention
raw_df_2.columns = ["Date", "Quarter", "Signal", "Entry_Price"]

# Handle index
raw_df_2["Date"] = pd.to_datetime(raw_df_2["Date"], errors="coerce", dayfirst=True)
raw_df_2["Date"] = raw_df_2["Date"] + pd.offsets.BDay(1)
raw_df_2 = raw_df_2.dropna(subset=["Date"])
raw_df_2 = raw_df_2.set_index("Date") 
raw_df_2 = raw_df_2.sort_index() 

# Change string signal to polar signal 
mapping = {np.nan:0, "BULLISH":1, "BEARISH":-1} 
raw_df_2["Clean_Signal"] = raw_df_2["Signal"].map(mapping)
raw_df_2["Clean_Signal"] = raw_df_2["Clean_Signal"].ffill().fillna(0).astype(int) 
raw_df_2.drop(columns=["Signal"], inplace = True)
raw_df_2.rename(columns={"Clean_Signal":"Signal"}, inplace=True)
raw_df_2 = raw_df_2[["Quarter", "Signal", "Entry_Price"]]

# Fix calendar
cme = mcal.get_calendar("CME_Equity")
schedule = cme.schedule(start_date=raw_df_2.index.min(),
                        end_date=raw_df_2.index.max())
trading_days = schedule.index
raw_df_2 = raw_df_2.reindex(trading_days)

# Save to csv
raw_df_2.reset_index().to_csv(r".\Signal_data\Q2_data_cleaned.csv", index=False)

# Preview
raw_df_2

,Quarter,Signal,Entry_Price
2026-03-30,2020Q2,0,6380.00
2026-03-31,2020Q2,1,6385.25
2026-04-01,2020Q2,1,6562.00
2026-04-02,2020Q2,1,6618.50
2026-04-03,2020Q2,-1,6622.50
2026-04-06,2020Q2,1,6590.00
2026-04-07,2020Q2,1,6650.50
2026-04-08,2020Q2,1,6689.00
2026-04-09,2020Q2,1,6814.25
2026-04-10,2020Q2,1,6858.00
